# LBPH Face Recognition

Pipeline: **Haar-Cascade** (face detection) → crop, resize, histogram-equalize → **LBPH** (face recognition).

## 1. Imports

In [ ]:
import json
import urllib.request
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split

## 2. Configuration

In [ ]:
# Project root works whether the notebook runs from impl/ or the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "impl" else Path.cwd()


@dataclass(frozen=True)
class Config:
    # Paths
    train_dir: Path = ROOT / "data" / "train"
    test_dir: Path = ROOT / "data" / "test"
    model_dir: Path = ROOT / "result" / "model"
    cascade_dir: Path = ROOT / "result" / "cascade"

    # Haar-Cascade detection
    cascade_file: str = "haarcascade_frontalface_default.xml"
    scale_factor: float = 1.2
    min_neighbors: int = 5
    min_face_size: tuple = (30, 30)

    # Preprocessing
    face_size: tuple = (200, 200)  # every face is resized to this before LBPH

    # LBPH
    radius: int = 4
    neighbors: int = 8
    grid_x: int = 8
    grid_y: int = 8
    unknown_threshold: float | None = 115.0  # None

    # Validation split
    val_size: float = 0.2
    random_state: int = 42

    @property
    def model_path(self) -> Path:
        return self.model_dir / "fr_model.yml"

    @property
    def labels_path(self) -> Path:
        return self.model_dir / "labels.json"


CFG = Config()
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CASCADE_URL = "https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/"

## 3. Face detection & preprocessing

In [ ]:
class FaceDetector:
    def __init__(self, cfg: Config = CFG):
        self.cfg = cfg
        self.cascade = cv2.CascadeClassifier(str(self._cascade_path()))
        if self.cascade.empty():
            raise RuntimeError(f"Could not load cascade: {cfg.cascade_file}")

    def _cascade_path(self) -> Path:
        """Use the XML bundled with OpenCV; newer wheels (5.x) omit it, so download once and cache."""
        bundled = (
            Path(getattr(getattr(cv2, "data", None), "haarcascades", ""))
            / self.cfg.cascade_file
        )
        if bundled.is_file():
            return bundled
        cached = self.cfg.cascade_dir / self.cfg.cascade_file
        if not cached.is_file():
            cached.parent.mkdir(parents=True, exist_ok=True)
            print(f"Downloading {self.cfg.cascade_file} ...")
            urllib.request.urlretrieve(CASCADE_URL + self.cfg.cascade_file, cached)
        return cached

    def detect_all(self, gray: np.ndarray) -> list:
        """Return every face as (x, y, w, h), sorted left to right."""
        faces = self.cascade.detectMultiScale(
            gray,
            scaleFactor=self.cfg.scale_factor,
            minNeighbors=self.cfg.min_neighbors,
            minSize=self.cfg.min_face_size,
        )
        return sorted((tuple(map(int, r)) for r in faces), key=lambda r: r[0])

    def detect(self, gray: np.ndarray):
        """Return the largest face as (x, y, w, h), or None."""
        faces = self.detect_all(gray)
        return max(faces, key=lambda r: r[2] * r[3]) if faces else None

    def preprocess(self, face_gray: np.ndarray) -> np.ndarray:
        face = cv2.resize(face_gray, self.cfg.face_size, interpolation=cv2.INTER_AREA)
        return cv2.equalizeHist(face)

    def extract_all(self, img_bgr: np.ndarray) -> list:
        """Return [(preprocessed_face, rect), ...] for every face in the image."""
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        return [
            (self.preprocess(gray[y : y + h, x : x + w]), (x, y, w, h))
            for x, y, w, h in self.detect_all(gray)
        ]

    def extract(self, img_bgr: np.ndarray):
        """Largest face only (used for training, where each image is one person).
        Return (preprocessed_face, rect) or (None, None) if no face is found."""
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        rect = self.detect(gray)
        if rect is None:
            return None, None
        x, y, w, h = rect
        return self.preprocess(gray[y : y + h, x : x + w]), rect


detector = FaceDetector()

## 4. Dataset loading
Expected layout: `data/<split>/<person name>/<image>`. The folder name is the label.

In [ ]:
def load_dataset(folder: Path, detector: FaceDetector, names: list[str] | None = None):
    """Load faces from folder. If names is given, labels follow that list (used for the test set)."""
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f"Dataset folder not found: {folder}")

    people = sorted(
        p for p in folder.iterdir() if p.is_dir()
    )  # sorted -> stable labels

    names = names or [p.name for p in people]
    faces, labels, skipped = [], [], []

    for person in people:
        if person.name not in names:
            print(f"Skipping unknown person folder: {person.name}")
            continue

        label = names.index(person.name)
        for img_path in sorted(person.iterdir()):
            if img_path.suffix.lower() not in IMG_EXTS:
                continue

            img = cv2.imread(str(img_path))
            face = detector.extract(img)[0] if img is not None else None
            if face is None:
                skipped.append(img_path)
                continue

            faces.append(face)
            labels.append(label)

    print(
        f"{folder.name}: {len(faces)} faces loaded, {len(skipped)} images skipped (no face/unreadable)"
    )

    return faces, np.array(labels, dtype=np.int32), names, skipped

## 5. Recognizer

In [ ]:
class FaceRecognizer:
    UNKNOWN = "Unknown"

    def __init__(self, cfg: Config = CFG):
        self.cfg = cfg
        self.model = cv2.face.LBPHFaceRecognizer_create(
            radius=cfg.radius,
            neighbors=cfg.neighbors,
            grid_x=cfg.grid_x,
            grid_y=cfg.grid_y,
        )
        self.names: list[str] = []

    def train(self, faces, labels, names):
        self.model.train(faces, np.asarray(labels, dtype=np.int32))
        self.names = list(names)
        return self

    def predict(self, face):
        """Return (label, name, distance). Lower distance = more similar."""
        label, distance = self.model.predict(face)
        t = self.cfg.unknown_threshold
        name = self.UNKNOWN if t is not None and distance > t else self.names[label]
        return label, name, distance

    def save(self):
        self.cfg.model_dir.mkdir(parents=True, exist_ok=True)  # overwrites old files
        self.model.write(str(self.cfg.model_path))
        self.cfg.labels_path.write_text(
            json.dumps(self.names, indent=2), encoding="utf-8"
        )
        print(f"Saved model -> {self.cfg.model_path}")

    @classmethod
    def load(cls, cfg: Config = CFG):
        if not (cfg.model_path.exists() and cfg.labels_path.exists()):
            raise FileNotFoundError(
                "No saved model found. Run the training section first."
            )
        rec = cls(cfg)
        rec.model.read(str(cfg.model_path))
        rec.names = json.loads(cfg.labels_path.read_text(encoding="utf-8"))
        return rec

## 6. Evaluation & visualization helpers

In [ ]:
def evaluate(rec: FaceRecognizer, faces, labels, title="Evaluation", verbose=False):
    preds, dists = [], []
    for face, actual in zip(faces, labels):
        pred, name, dist = rec.predict(face)
        preds.append(pred)
        dists.append(dist)
        if verbose:
            mark = "OK " if pred == actual else "BAD"
            print(
                f"[{mark}] actual={rec.names[actual]:<15} predicted={name:<15} distance={dist:.2f}"
            )

    preds, dists = np.array(preds), np.array(dists)
    acc = accuracy_score(labels, preds)
    print(f"\n=== {title} ===")
    print(f"Accuracy: {acc:.2%} ({(preds == labels).sum()}/{len(labels)})")
    print(
        f"Distance: mean={dists.mean():.1f}, correct mean={dists[preds == labels].mean():.1f}"
    )
    ids = list(range(len(rec.names)))
    print(
        classification_report(
            labels, preds, labels=ids, target_names=rec.names, zero_division=0
        )
    )

    _, ax = plt.subplots(figsize=(7, 7))
    ConfusionMatrixDisplay.from_predictions(
        labels,
        preds,
        labels=ids,
        display_labels=rec.names,
        xticks_rotation=45,
        ax=ax,
        colorbar=False,
    )
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return acc


def show_image(img_bgr, title=""):
    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()


def annotate(img_bgr, rect, text, color=(0, 0, 255)):
    x, y, w, h = rect
    scale = max(0.5, w / 250)  # text scales with face size
    cv2.rectangle(img_bgr, (x, y), (x + w, y + h), color, 2)
    cv2.putText(
        img_bgr, text, (x, max(y - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 2
    )
    return img_bgr

## 7. Train & evaluate
Trains on `data/train` (minus a validation split), then checks the held-out `data/test` set. Saved files are overwritten on every run.

In [ ]:
faces, labels, names, skipped = load_dataset(CFG.train_dir, detector)
print("Labels:", dict(enumerate(names)))

x_train, x_val, y_train, y_val = train_test_split(
    faces,
    labels,
    test_size=CFG.val_size,
    random_state=CFG.random_state,
    stratify=labels,
)
print(f"Train: {len(x_train)} | Validation: {len(x_val)}")

recognizer = FaceRecognizer().train(x_train, y_train, names)
evaluate(recognizer, x_val, y_val, title="Validation")

In [ ]:
test_faces, test_labels, _, _ = load_dataset(CFG.test_dir, detector, names=names)
evaluate(recognizer, test_faces, test_labels, title="Test", verbose=True)

In [ ]:
# Retrain on all training data (train + validation) before saving
recognizer = FaceRecognizer().train(faces, labels, names)
recognizer.save()

## 8. Predict on an image
Detects and recognizes **every** face in the image (group photos work). Uses the saved model, so this works without re-training.

Note: LBPH always returns the closest of the trained people, so strangers in a group photo get a wrong name unless `unknown_threshold` is set.

In [ ]:
def predict_image(image_path, rec: FaceRecognizer | None = None, show=True):
    rec = rec or FaceRecognizer.load()
    path = Path(image_path)
    img = cv2.imread(str(path)) if path.is_file() else None
    if img is None:
        print(f"Cannot read image: {path}")
        return None

    detections = detector.extract_all(img)
    if not detections:
        print("No face detected.")
        return []

    results, canvas = [], img.copy()
    for i, (face, rect) in enumerate(detections, 1):
        _, name, dist = rec.predict(face)
        results.append({"name": name, "distance": dist, "box": rect})
        print(f"Face {i}: {name:<15} distance={dist:.2f} box={rect}")
        color = (0, 0, 255) if name == rec.UNKNOWN else (0, 200, 0)  # red = unknown
        annotate(canvas, rect, f"{name} ({dist:.1f})", color)

    if show:
        show_image(canvas, title=f"{path.name}: {len(results)} face(s)")
    return results


predict_image(CFG.test_dir / "Joji" / "1.jpg")

## 9. Interactive menu

In [ ]:
def main_menu():
    while True:
        choice = input("\n1. Predict image  2. Exit\nChoice: ").strip()
        if choice == "1":
            predict_image(input("Image path: ").strip().strip('"'))
        elif choice == "2":
            break
        else:
            print("Invalid choice!")


# main_menu()